# trio-retina — quickstart: detector → events in 60s

**What you'll see:** a synthetic detector walks one `person` across a frame, and Retina emits `count` / `zone.enter` / `zone.dwell` / `line.cross` events as standard `retina.event` JSON — then `validate()` confirms the schema.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machinefi/trio-retina/blob/main/notebooks/01_quickstart_events.ipynb)


Install from PyPI — pure Python + numpy, no model or GPU needed.

In [ ]:
%pip install trio-retina

## A scripted "model"

Any `callable(frame) -> list[Detection]` is a detector. Here we fake one that marches a single `person` box left-to-right (swap in `YoloDetector("yolo11n.pt")` for the real thing).

In [ ]:
import numpy as np
from retina import CountRule, IoUTracker, Line, LineRule, Retina, Zone, ZoneRule
from retina.detect import Detection


class ScriptedDetector:
    """Returns one 'person' box marching across the frame, one step per call."""

    def __init__(self):
        self._xs = list(range(0, 102, 6))
        self._i = 0

    def __call__(self, frame: np.ndarray) -> list[Detection]:
        if self._i >= len(self._xs):
            return []
        x = self._xs[self._i]
        self._i += 1
        return [Detection(label="person", bbox=(x - 10, 40, x + 10, 60), confidence=0.9)]

## Wire detector → tracker → rules and run

A zone (the dock), a tripwire line (the door), and a count rule. `Retina.run(frames)` yields one `Event` per fired rule — we print each as JSON.

In [ ]:
dock = Zone("dock", [(40, 0), (60, 0), (60, 100), (40, 100)])
tripwire = Line("door", (50, 0), (50, 100))

cam = Retina(
    source_id="cam_01",
    detector=ScriptedDetector(),
    tracker=IoUTracker(min_hits=2),
    rules=[
        ZoneRule(dock, classes={"person"}, dwell_s=2.0),
        LineRule(tripwire, classes={"person"}),
        CountRule(threshold=1, classes={"person"}),
    ],
)

frames = [(np.zeros((100, 100, 3), dtype=np.uint8), float(i)) for i in range(18)]
events = list(cam.run(frames))
for ev in events:
    print(ev.to_json())

## The key part: it's a validated, serializable event stream

Every event conforms to the `retina.event` schema. `validate()` returns an empty list when an event is valid — that contract is what lets the next layer (dynamics / digital twin / agent) consume it.

In [ ]:
from retina import validate

ev = events[0]
print(ev.to_json())
print("validate(ev) ==", validate(ev))  # [] means valid